# 8.11 - RAG Evaluation & RAGAS

**Module 8 - RAG Systems** | Netsetos GenAI Engineering

This notebook works through RAG Evaluation & RAGAS hands-on: The four RAG metrics, built from scratch; RAGAS: the same metrics, batched on the current API; Generate a synthetic golden test set; The improvement loop: A/B test every change; DeepEval: evaluation as a pytest gate; Rank-aware retrieval metrics in pure Python.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - a note on reproducibility

A single comment cell flagging that the notebook's evaluation runs aim to be as repeatable as possible. Because the metrics lean on LLM-judge calls, the code pins the judge to temperature 0 downstream - this comment is just the heads-up that scores still wobble slightly between runs.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a comment-only cell with no executable logic. It exists to set expectations: LLM-judged metrics are non-deterministic, so the notebook does what it can (temperature 0, fixed prompts) to keep results stable.

**How the code works - step by step**
- The entire cell is a single `#` comment - nothing runs.
- It signals the reproducibility posture for the judge calls that follow.

**In one line:** a note, not code.

**What you'll see:** (no output - it's a comment)

## Setup - install the evaluation stack

One commented `pip install` line with every third-party package the lesson touches. Uncomment it on Colab or a fresh environment; skip it if these are already installed.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install google-genai langchain-core langchain-google-genai deepeval ragas python-dotenv -q  # noqa


**Explanation**

Environment prep only - it installs the SDKs the later cells import. The line is commented out so a re-run of an already-provisioned kernel does nothing.

**How the code works - step by step**
- **`google-genai`** - the current Google Gemini SDK used for the by-hand judge calls.
- **`langchain-core` / `langchain-google-genai`** - LangChain wrappers RAGAS uses to drive Gemini as judge and embedder.
- **`ragas`** - the automated RAG-evaluation library (v0.2+ API).
- **`deepeval`** - evaluation-as-pytest, for the CI-gate step.
- **`python-dotenv`** - loads API keys from a `.env` file.

**What you'll see:** (no output - environment prep; the install line is commented out)

## Setup - load your API key

Reads your Google AI Studio key from the environment rather than hardcoding it. Any one provider key is enough to start the by-hand steps.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A credentials step, not a model call. `setdefault` seeds `GOOGLE_API_KEY` only if it is not already set, so a key exported in your shell or loaded from `.env` wins.

**How the code works - step by step**
- Imports `os`.
- `os.environ.setdefault("GOOGLE_API_KEY", "")` - leaves an existing key untouched; otherwise sets an empty placeholder you fill in.
- The comment points you at aistudio.google.com to get the key.

**In one line:** make the Gemini key available without writing it into the notebook.

**What you'll see:** (no output - it just prepares the environment variable)

## 1 - The four RAG metrics, built from scratch

Before trusting a framework's number, build it yourself. Each of the four core metrics - faithfulness, answer relevance, context precision, context recall - is a single LLM-judge call with a specific prompt and a 0-1 score. Writing them once makes RAGAS legible.

> **Needs a Google AI Studio key** (set GOOGLE_API_KEY)

In [ ]:
# THE 4 RAG METRICS, BUILT FROM SCRATCH - one LLM-judge call each
# pip install google-genai
import json, re
from google import genai
from google.genai import types

client = genai.Client()   # reads GOOGLE_API_KEY

def judge(prompt):
    # temperature 0 makes the judge as deterministic as possible
    txt = client.models.generate_content(model="gemini-2.5-flash", contents=prompt,
        config=types.GenerateContentConfig(temperature=0)).text
    return json.loads(re.sub(r"```(json)?", "", txt).strip())

class RAGEvaluator:
    def faithfulness(self, answer, contexts):
        # supported claims / total claims - catches hallucination
        ctx = "\n".join(f"[{i+1}] {c}" for i, c in enumerate(contexts))
        d = judge(f"Break the answer into claims; mark each supported by the context or not. "
                  f'Return JSON {{"faithfulness_score":0-1}}.\nAnswer: {answer}\nContext:\n{ctx}')
        return float(d.get("faithfulness_score", 0.5))

    def answer_relevance(self, question, answer):
        d = judge(f'Rate 0-1 how well the answer addresses the question. Return JSON '
                  f'{{"relevance_score":0-1}}.\nQ: {question}\nA: {answer}')
        return float(d.get("relevance_score", 0.5))

    def context_precision(self, question, contexts):
        # relevant retrieved / total retrieved - retrieval noise
        ctx = "\n".join(f"[{i+1}] {c[:200]}" for i, c in enumerate(contexts))
        d = judge(f'For each context, is it relevant to the question? Return JSON '
                  f'{{"precision_score":0-1}}.\nQ: {question}\nContexts:\n{ctx}')
        return float(d.get("precision_score", 0.5))

    def context_recall(self, ground_truth, contexts):
        # ground-truth facts found in contexts / total - missed context
        ctx = "\n".join(f"[{i+1}] {c[:200]}" for i, c in enumerate(contexts))
        d = judge(f'Break the ground truth into facts; is each found in the contexts? Return JSON '
                  f'{{"recall_score":0-1}}.\nGround truth: {ground_truth}\nContexts:\n{ctx}')
        return float(d.get("recall_score", 0.5))

    def evaluate(self, question, answer, contexts, ground_truth=""):
        s = {"faithfulness": self.faithfulness(answer, contexts),
             "answer_relevance": self.answer_relevance(question, answer),
             "context_precision": self.context_precision(question, contexts)}
        if ground_truth: s["context_recall"] = self.context_recall(ground_truth, contexts)
        s["overall"] = sum(s.values()) / len(s)
        return s

result = RAGEvaluator().evaluate(
    question="What is the refund policy?",
    answer="Refunds are processed within 14 business days via form RF-201. The company was founded in 2020.",
    contexts=["Refunds processed within 14 business days. Submit form RF-201 with your order ID.",
              "The platform supports Python 3.9+ and requires Docker."],   # the 2nd chunk is noise
    ground_truth="Refunds take 14 business days. Form RF-201 required with the order ID.")
for metric, score in result.items():
    print(f"  {metric:20s} {'#'*int(score*20):20s} {score:.2f}")

# Output: (scores vary slightly by judge run)
#   faithfulness         #############        0.67  <- "founded in 2020" is NOT in context = hallucination
#   answer_relevance     ##################   0.90  <- directly answers the refund question
#   context_precision    ##########           0.50  <- only 1 of 2 chunks relevant (Docker chunk was noise)
#   context_recall       ####################  1.00 <- all ground-truth facts were found
#   overall              ##############       0.77

**Explanation**

The whole class is four judge calls wrapped around one helper. Each method builds a prompt, sends it to a temperature-0 Gemini judge, and parses a JSON score; `evaluate` runs them and averages. Read it as: this is a measurement harness that scores one Q&A four different ways, not a RAG pipeline itself.

**How the code works - step by step**
- **`judge(prompt)`** - calls `gemini-2.5-flash` at temperature 0, strips any ```json fences with a regex, and `json.loads` the result into a dict.
- **`faithfulness`** - numbers the contexts, asks the judge to break the answer into claims and mark each supported or not, returns `faithfulness_score` (catches hallucination).
- **`answer_relevance`** - asks the judge to rate 0-1 how well the answer addresses the question.
- **`context_precision`** - asks whether each retrieved chunk is relevant to the question (retrieval noise).
- **`context_recall`** - breaks the ground truth into facts and checks each appears in the contexts (missed context).
- **`evaluate`** - always runs the first three, adds recall only if `ground_truth` is passed, then appends an averaged `overall`.
- The demo scores a refund Q&A whose answer plants one hallucination ("founded in 2020") and whose contexts include one noise chunk (the Docker line).

**In one line:** four judge calls, one JSON score each, averaged into an overall.

**What you'll see:** An ASCII bar chart of five metrics: faithfulness ~0.67 (flags "founded in 2020" as unsupported), answer_relevance ~0.90, context_precision ~0.50 (the Docker chunk is noise), context_recall 1.00, overall ~0.77. Scores vary slightly per judge run.

## 2 - RAGAS: the same metrics, batched on the current API

Now that you know what the metrics compute, let the library do it at scale. RAGAS handles batching, retries, and a per-row scorecard - but its API changed. The current v0.2+ shape uses `EvaluationDataset.from_list`, renamed fields, and metric classes you instantiate.

> **Needs a Google AI Studio key** (set GOOGLE_API_KEY)

In [ ]:
# RAGAS LIBRARY - the CURRENT API (EvaluationDataset + metric classes)
# pip install ragas langchain-google-genai
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithReference, LLMContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

judge = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0))
emb = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

# each row uses the CURRENT field names (question->user_input, answer->response,
# contexts->retrieved_contexts, ground_truth->reference)
samples = [
    {"user_input": "What is the refund policy?",
     "retrieved_contexts": ["Refunds processed within 14 business days. Submit form RF-201 with order ID."],
     "response": "Refunds are processed within 14 business days via form RF-201.",
     "reference": "Refunds take 14 business days. Form RF-201 required with order ID."},
    {"user_input": "What GPU is required?",
     "retrieved_contexts": ["GPU acceleration requires CUDA 12.1+. Minimum 8 GB VRAM."],
     "response": "You need CUDA 12.1+ and 8 GB VRAM minimum.",
     "reference": "CUDA 12.1 or later. 8 GB VRAM minimum."},
]
dataset = EvaluationDataset.from_list(samples)

result = evaluate(
    dataset=dataset,
    metrics=[Faithfulness(), ResponseRelevancy(),
             LLMContextPrecisionWithReference(), LLMContextRecall()],
    llm=judge, embeddings=emb,
)
print(result)                 # aggregate scores across the dataset
print(result.to_pandas())     # per-sample breakdown as a DataFrame

# Output:
# {'faithfulness': 1.00, 'answer_relevancy': 0.94, 'llm_context_precision_with_reference': 1.00, 'context_recall': 1.00}
# (plus a per-row DataFrame: swap a chunking strategy, re-run, compare the columns)

**Explanation**

This cell swaps your hand-built harness for the official library while proving the API migration. It wraps Gemini as both the judge LLM and the embedder, builds a dataset of rows with the CURRENT field names, and runs the four metric classes over it. Read it as: the industrial version of step 1, where the class names map straight back to your four metrics.

**How the code works - step by step**
- **`LangchainLLMWrapper(ChatGoogleGenerativeAI(...))`** - wraps Gemini 2.5 Flash at temperature 0 as the judge.
- **`LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(...))`** - wraps the Gemini embedding model that relevancy needs.
- **`samples`** - rows using the renamed fields: `user_input` (was question), `retrieved_contexts` (was contexts), `response` (was answer), `reference` (was ground_truth).
- **`EvaluationDataset.from_list(samples)`** - the replacement for the deprecated `Dataset.from_dict`.
- **`evaluate(...)`** - runs `Faithfulness()`, `ResponseRelevancy()`, `LLMContextPrecisionWithReference()`, `LLMContextRecall()` (metric CLASSES) with the wrapped LLM and embeddings.
- The two prints show the aggregate scores and the per-sample DataFrame.

**In one line:** current-API RAGAS runs your four metrics over a dataset and returns aggregate + per-row scores.

**What you'll see:** An aggregate dict roughly `{'faithfulness': 1.00, 'answer_relevancy': 0.94, 'llm_context_precision_with_reference': 1.00, 'context_recall': 1.00}` plus a per-row DataFrame you can diff after a pipeline change.

## 3 - Generate a synthetic golden test set

Metrics are useless without questions to run them on, and hand-writing 200 is slow. RAGAS's `TestsetGenerator` builds a knowledge graph from your documents and synthesizes diverse Q&A pairs - single-hop and multi-hop - each with a reference answer.

> **Needs a Google AI Studio key** (set GOOGLE_API_KEY)

In [ ]:
# SYNTHETIC TEST DATA - RAGAS TestsetGenerator (knowledge-graph based, v0.2+)
# pip install ragas langchain-google-genai
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

gen_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-2.5-flash"))
gen_emb = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

docs = [Document(page_content=c) for c in [
    "The NetsAI course costs 29,999 rupees/year or 2,999/month. Enterprise from 4,99,000/year.",
    "Refunds are processed within 14 business days. Submit form RF-201 with your order ID.",
    "GPU acceleration requires CUDA 12.1+ and 8 GB VRAM minimum."]]

generator = TestsetGenerator(llm=gen_llm, embedding_model=gen_emb)
testset = generator.generate_with_langchain_docs(docs, testset_size=6)
# default query mix: SingleHopSpecific 0.5, MultiHopAbstract 0.25, MultiHopSpecific 0.25

df = testset.to_pandas()
print(df[["user_input", "reference"]].head())

# Output: (KG-synthesized Q&A, ready to feed step 2's evaluate())
# user_input                                       reference
# What form is needed to request a refund?         Form RF-201, submitted with the order ID.
# What are the GPU requirements for the course?    CUDA 12.1 or later and at least 8 GB VRAM.
# ... 6 rows: a mix of single-hop and multi-hop questions

**Explanation**

A test-data factory, not an evaluation. It feeds a few LangChain `Document`s into RAGAS, which constructs a knowledge graph and drafts questions of varying hop-count. The output is a ready evaluation set you can hand straight to step 2's `evaluate()`.

**How the code works - step by step**
- **`gen_llm` / `gen_emb`** - the wrapped Gemini LLM and embedder that drive generation.
- **`docs`** - three source documents (pricing, refunds, GPU requirements) wrapped as `Document(page_content=...)`.
- **`TestsetGenerator(llm=, embedding_model=)`** - builds the knowledge graph over those docs.
- **`generate_with_langchain_docs(docs, testset_size=6)`** - synthesizes 6 Q&A pairs; the comment notes the default query mix (single-hop specific 0.5, multi-hop abstract 0.25, multi-hop specific 0.25).
- **`testset.to_pandas()`** - materializes the rows; the print shows the `user_input` / `reference` columns.

**In one line:** documents in, a mixed single-/multi-hop Q&A set out.

**What you'll see:** A DataFrame head of ~6 synthesized rows - questions like "What form is needed to request a refund?" paired with reference answers - a mix of single-hop and multi-hop questions ready for `evaluate()`.

## 4 - The improvement loop: A/B test every change

Metrics only earn their keep when they drive decisions. The loop is: measure a baseline, change one thing, re-evaluate on the SAME test set, deploy only if scores rose. This turns every RAG tweak into an experiment.

In [ ]:
# THE IMPROVEMENT LOOP - A/B test RAG changes with the metrics from step 1
class RAGExperiment:
    def __init__(self, eval_set):
        self.eval_set, self.evaluator, self.results = eval_set, RAGEvaluator(), {}

    def run(self, name, rag_fn):
        # run a pipeline on every test question, score it, average each metric over the rows that have it
        rows = [self.evaluator.evaluate(qa["question"], (r := rag_fn(qa["question"]))["answer"],
                                        r["contexts"], qa.get("answer", "")) for qa in self.eval_set]
        self.results[name] = {k: sum(r[k] for r in rows if k in r) / sum(1 for r in rows if k in r) for k in rows[0]}
        return self.results[name]

    def compare(self):
        names = list(self.results)
        for m in self.results[names[0]]:
            vals = [self.results[n][m] for n in names]
            delta = vals[-1] - vals[0]
            arrow = "up" if delta > 0.02 else ("down" if delta < -0.02 else "tie")
            print(f"  {m:20s} " + " ".join(f'{v:.2f}' for v in vals) + f"   {arrow} {delta:+.2f}")
        best = max(names, key=lambda n: self.results[n]["overall"])
        print(f"  winner: {best}")

test_set = [{"question": "What is the refund policy?", "answer": "14 business days, form RF-201."}]
exp = RAGExperiment(test_set)
exp.run("v1_basic", lambda q: {"answer": "Refunds take 14 days.", "contexts": ["Refunds within 14 business days."]})
exp.run("v2_grounded", lambda q: {"answer": "Refunds take 14 business days; submit form RF-201.", "contexts": ["Refunds within 14 business days. Submit RF-201."]})
exp.compare()

# Output:
#   faithfulness         1.00 1.00   tie +0.00
#   answer_relevance     0.80 0.95   up  +0.15   <- v2 answers more completely
#   context_precision    1.00 1.00   tie +0.00
#   context_recall       0.60 1.00   up  +0.40   <- v2's context covers form RF-201
#   winner: v2_grounded

**Explanation**

A harness that reuses the step-1 evaluator to score two RAG configurations on one test set and print the deltas. No new metrics here - it's the discipline layer: run, compare, pick the winner. Note it calls `RAGEvaluator` from cell 4, so that cell must have run first.

**How the code works - step by step**
- **`__init__`** - stores the eval set, a fresh `RAGEvaluator`, and an empty results dict.
- **`run(name, rag_fn)`** - runs the pipeline on every question, scores each with the evaluator, and averages each metric only over the rows that have it (recall is optional).
- **`compare`** - prints each metric across all experiments, computes the last-minus-first delta, and labels it up / down / tie against a 0.02 threshold, then names the winner by `overall`.
- The demo registers `v1_basic` (terse answer, thin context) and `v2_grounded` (fuller answer, richer context) and compares them.

**In one line:** score two configs on one set, show deltas, treat sub-0.02 moves as ties.

**What you'll see:** A side-by-side table: faithfulness tie +0.00, answer_relevance up +0.15, context_precision tie +0.00, context_recall up +0.40, and `winner: v2_grounded`.

## 5 - DeepEval: evaluation as a pytest gate

Production teams want evaluation in the SAME place as their other tests - the CI pipeline. DeepEval treats every answer as an assertion you can fail a build on, and `GEval` turns any plain-English criterion into a metric.

> **Needs an LLM-judge key** - DeepEval defaults to OpenAI (set OPENAI_API_KEY). Run with `deepeval test run test_rag.py`.

In [ ]:
# DEEPEVAL - evaluation as pytest; gate the CI build on RAG quality
# pip install deepeval  ;  run with:  deepeval test run test_rag.py
from deepeval import assert_test
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric, ContextualRelevancyMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

def test_refund_answer():
    case = LLMTestCase(
        input="What is the refund policy?",
        actual_output="Refunds are processed within 14 business days via form RF-201.",
        retrieval_context=["Refunds processed within 14 business days. Submit form RF-201."])

    # the referenceless RAG triad (faithfulness + answer + context relevancy) - no gold answer needed
    faithfulness = FaithfulnessMetric(threshold=0.8)
    relevancy = AnswerRelevancyMetric(threshold=0.8)
    context_rel = ContextualRelevancyMetric(threshold=0.7)   # the third triad member: is the retrieved context relevant?
    # a CUSTOM metric from a plain-English criterion
    conciseness = GEval(name="Conciseness", threshold=0.7,
        criteria="Is the answer concise and free of filler?",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT])

    assert_test(case, [faithfulness, relevancy, context_rel, conciseness])   # red build if any metric < threshold

# Output: (deepeval test run)
# test_rag.py::test_refund_answer PASSED
#   Faithfulness 1.00 | Answer Relevancy 0.94 | Contextual Relevancy 0.90 | Conciseness 0.88   (all pass)

**Explanation**

This is a pytest test function, not a script you run top-to-bottom. It packages one RAG answer as an `LLMTestCase`, attaches the referenceless RAG triad plus a custom conciseness metric, and asserts them all - a red build if any metric falls below its threshold.

**How the code works - step by step**
- **`LLMTestCase`** - holds the `input`, the `actual_output`, and the `retrieval_context` (no gold answer required).
- **`FaithfulnessMetric` / `AnswerRelevancyMetric` / `ContextualRelevancyMetric`** - the referenceless triad, each with its own threshold.
- **`GEval(name="Conciseness", ...)`** - a custom metric built from the plain-English criterion "is the answer concise and free of filler?", scored on `ACTUAL_OUTPUT`.
- **`assert_test(case, [...])`** - fails the build if any metric scores below its threshold.

**In one line:** a bad answer becomes a failing pytest assertion.

**What you'll see:** Under `deepeval test run`: `test_rag.py::test_refund_answer PASSED` with Faithfulness 1.00, Answer Relevancy 0.94, Contextual Relevancy 0.90, Conciseness 0.88 - all above threshold.

## 6 - Rank-aware retrieval metrics in pure Python

RAGAS's context precision/recall are LLM-judged and order-blind, but rank matters enormously - a relevant doc at position 1 is worth far more than the same doc at position 8. MRR and NDCG catch ordering problems RAGAS misses, cheaply and deterministically.

In [ ]:
# RETRIEVAL METRICS FROM SCRATCH - evaluate the RETRIEVER, no LLM judge, no noise
import math

def precision_at_k(retrieved, relevant, k):
    return sum(d in relevant for d in retrieved[:k]) / k

def recall_at_k(retrieved, relevant, k):
    return sum(d in relevant for d in retrieved[:k]) / len(relevant)

def mrr(retrieved, relevant):
    for i, d in enumerate(retrieved, 1):
        if d in relevant: return 1.0 / i        # reciprocal rank of the FIRST relevant doc
    return 0.0

def ndcg_at_k(retrieved, gains, k):
    def dcg(order):
        return sum(gains.get(d, 0) / math.log2(i + 1) for i, d in enumerate(order[:k], 1))
    ideal = sorted(gains, key=gains.get, reverse=True)   # best possible ranking
    return dcg(retrieved) / dcg(ideal) if dcg(ideal) else 0.0

retrieved = ["d3", "d1", "d7", "d2"]            # the ranked results
relevant  = {"d1", "d2"}                         # binary relevant set
gains     = {"d1": 3, "d2": 2, "d3": 0, "d7": 1}   # graded relevance

print(f"Precision@2: {precision_at_k(retrieved, relevant, 2):.2f}")
print(f"Recall@4:    {recall_at_k(retrieved, relevant, 4):.2f}")
print(f"MRR:         {mrr(retrieved, relevant):.2f}")
print(f"NDCG@4:      {ndcg_at_k(retrieved, gains, 4):.2f}")

# Output:
# Precision@2: 0.50   (1 of the top-2 is relevant)
# Recall@4:    1.00   (both relevant docs are in the top-4)
# MRR:         0.50   (first relevant doc is at rank 2 -> 1/2)
# NDCG@4:      0.68   (relevant docs are present but poorly ordered)

**Explanation**

Four small functions that score a ranked retrieval result against a labelled relevant set - no LLM, no noise, fully deterministic. Precision/recall count WHICH docs came back; MRR/NDCG care WHERE they ranked. This is the retrieval half of component isolation.

**How the code works - step by step**
- **`precision_at_k`** - fraction of the top-k that is relevant.
- **`recall_at_k`** - fraction of all relevant docs found in the top-k.
- **`mrr`** - reciprocal rank of the FIRST relevant doc (1/rank), great for single-answer QA.
- **`ndcg_at_k`** - discounts each doc's graded relevance by `log2(rank+1)`, then normalizes by the ideal (best-possible) ordering - the gold standard.
- The demo ranks `[d3, d1, d7, d2]` with binary relevant `{d1, d2}` and graded gains, then prints all four.

**In one line:** count relevance (precision/recall) and reward good ordering (MRR/NDCG).

**What you'll see:** Precision@2 0.50, Recall@4 1.00, MRR 0.50 (first relevant doc at rank 2), NDCG@4 0.68 - the right docs are present but poorly ordered.

## 7 - Production monitoring: alert on trends, not single readings

Passing your eval set proves the RAG was good the day you shipped. Production drifts - queries change, docs change, the model updates under you. Observability samples quality on live traffic and alerts when it trends off a rolling baseline.

In [ ]:
# PRODUCTION MONITORING - alert on quality/cost TRENDS, not single readings
def should_alert(faith_today, faith_7day_avg, cost_today, cost_7day_avg):
    alerts = []
    if faith_today < faith_7day_avg - 0.10:      # faithfulness drift
        alerts.append("FAITHFULNESS DROP")
    if cost_today > cost_7day_avg * 1.5:         # cost spike
        alerts.append("COST SPIKE")
    return alerts or ["ok"]

# platforms that trace + score live traffic: Langfuse (github.com/langfuse/langfuse,
# MIT, self-host), Arize Phoenix (OTEL-native, native RAGAS), LangSmith, TruLens.
print(should_alert(faith_today=0.79, faith_7day_avg=0.92, cost_today=180, cost_7day_avg=100))

# Output:
# ['FAITHFULNESS DROP', 'COST SPIKE']

**Explanation**

A tiny alerting rule that compares today's numbers against a 7-day average - the core logic every observability platform wraps. It fires only on meaningful drift, not on a single noisy reading. Pure Python, no judge call here.

**How the code works - step by step**
- **`should_alert(...)`** - takes today's faithfulness and cost plus their 7-day averages.
- Fires **"FAITHFULNESS DROP"** when today is more than 0.10 below the 7-day average.
- Fires **"COST SPIKE"** when today's cost exceeds 1.5x the 7-day average.
- Returns the collected alerts, or `["ok"]` if none.
- The comment lists the real platforms (Langfuse, Arize Phoenix, LangSmith, TruLens); the demo passes a drifted day (0.79 vs 0.92, 180 vs 100).

**In one line:** compare today to a rolling average and alert on drift.

**What you'll see:** `['FAITHFULNESS DROP', 'COST SPIKE']` - both thresholds are crossed by the demo inputs.

## 8 - Component isolation: name the broken half

The lesson's opening misconception - "high faithfulness means good RAG" - is really a plea for isolation. A single overall score hides which half is broken. Evaluate retrieval and generation SEPARATELY and the failure signature becomes unambiguous.

In [ ]:
# COMPONENT ISOLATION - the metric combination names the broken half
def diagnose(retrieval_score, faithfulness, answer_relevance):
    issues = []
    if retrieval_score < 0.7:
        issues.append("RETRIEVAL: wrong/missing chunks (fix chunking, top_k, reranking)")
    if faithfulness < 0.7:
        issues.append("GENERATION: hallucinating (stronger grounding prompt, lower temperature)")
    if retrieval_score >= 0.7 and faithfulness >= 0.9 and answer_relevance < 0.7:
        issues.append("PROMPT: grounded but off-topic (improve the answer template)")
    return issues or ["all components healthy"]

# case A: faithfulness HIGH but answers bad -> blame RETRIEVAL, not the LLM
print("case A:", diagnose(retrieval_score=0.4, faithfulness=0.92, answer_relevance=0.6))
# case B: retrieval good, faithfulness low -> the GENERATOR is hallucinating
print("case B:", diagnose(retrieval_score=0.9, faithfulness=0.5, answer_relevance=0.8))

# Output:
# case A: ['RETRIEVAL: wrong/missing chunks (fix chunking, top_k, reranking)']
# case B: ['GENERATION: hallucinating (stronger grounding prompt, lower temperature)']

**Explanation**

A diagnostic rule that reads a retrieval score alongside generation scores and names the failing component. It encodes the key insight: high faithfulness with bad answers is a RETRIEVAL bug (the model faithfully summarizes wrong chunks), not a generation one. Pure Python, no judge.

**How the code works - step by step**
- **`diagnose(retrieval_score, faithfulness, answer_relevance)`** - returns a list of issues.
- Retrieval < 0.7 -> flags RETRIEVAL (fix chunking, top_k, reranking).
- Faithfulness < 0.7 -> flags GENERATION (stronger grounding prompt, lower temperature).
- Good retrieval + very high faithfulness + low answer relevance -> flags PROMPT (grounded but off-topic).
- Returns `["all components healthy"]` if nothing trips.
- Demo case A (retrieval 0.4, faithfulness 0.92) and case B (retrieval 0.9, faithfulness 0.5) show the two classic signatures.

**In one line:** the combination of scores, not any single one, names the culprit.

**What you'll see:** `case A: ['RETRIEVAL: wrong/missing chunks (fix chunking, top_k, reranking)']` and `case B: ['GENERATION: hallucinating (stronger grounding prompt, lower temperature)']`.

## 9 - India legal eval: exact citation checking

Where LLM judges fail hardest - multilingual and high-stakes regulatory text. An Indian legal answer hinges on citation precision ("Section 17(5)(d)" vs "17(5)(c)") that changes the legal meaning entirely and that no fluency-based judge reliably verifies.

In [ ]:
# INDIA LEGAL EVAL - exact citation checking (a fluency-based judge misses this)
import re
CITATION = re.compile(r"Section\s+\d+\(\d+\)\([a-z]\)")

def citations_match(answer, reference):
    a, r = set(CITATION.findall(answer)), set(CITATION.findall(reference))
    return a == r, (r - a)      # exact match? and which references are missing/wrong

ok, missing = citations_match(
    answer="Input tax credit is blocked under Section 17(5)(c).",
    reference="Input tax credit is blocked under Section 17(5)(d).")   # ONE letter differs
print("citations match:", ok, "| missing/wrong:", missing)

# A fluent, grounded-sounding answer with the WRONG sub-clause is a legal error a judge rates as fine.
# The tradeoff: exact checks are strict and language-specific - when to use them is exactly here,
# high-stakes regulatory text, alongside a human expert.

# Output:
# citations match: False | missing/wrong: {'Section 17(5)(d)'}

**Explanation**

A deterministic regex check that no LLM judge can substitute for. It extracts every statutory citation from the answer and the reference and demands an exact set match - a fluent, grounded-sounding answer with one wrong sub-clause letter fails here even though a faithfulness judge would rate it fine.

**How the code works - step by step**
- **`CITATION`** - a compiled regex matching `Section N(N)(x)` citation patterns.
- **`citations_match(answer, reference)`** - extracts the citation set from each, returns whether they are equal and which reference citations are missing or wrong.
- The demo compares an answer citing `Section 17(5)(c)` against a reference of `Section 17(5)(d)` - one letter apart.
- The comment underscores the tradeoff: exact checks are strict and language-specific, reserved for high-stakes regulatory text alongside a human expert.

**In one line:** exact-match the citations; one wrong letter is a hard fail.

**What you'll see:** `citations match: False | missing/wrong: {'Section 17(5)(d)'}` - the single-letter difference is caught.

Read top to bottom, this notebook walks the whole evaluation stack: you first build the four core RAG metrics by hand as temperature-0 LLM-judge calls, then hand the same job to the current RAGAS v0.2+ API (EvaluationDataset + metric classes), generate a synthetic golden set, and A/B-test changes on a fixed test set. From there it shifts to the tools you ship with - DeepEval as pytest CI gates, rank-aware retrieval metrics (MRR/NDCG) in pure Python, trend-based production monitoring, component isolation that names the broken half, and India-specific exact-citation checking. The through-line is simple: every RAG decision should be backed by a number, every number checked against a second one (retrieval vs generation, judge vs human), and no single score is ever the whole truth. Next stops for these metrics: Module 9 (Fine-Tuning) uses them as the before/after scorecard, and Module 14 (LLMOps) wires the step-7 observability into CI gates and production dashboards.